In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load preprocessed data from data_preprocessing.ipynb
# These dataframes have been cleaned and processed
coaches_df        = pd.read_pickle('../merged_datasets/coaches_df.pkl')
teams_post_df     = pd.read_pickle('../merged_datasets/teams_post_df.pkl')
teams_df          = pd.read_pickle('../merged_datasets/teams_df.pkl')

print("✓ Loaded preprocessed datasets from data_preprocessing.ipynb")
print(f"  - coaches_df: {len(coaches_df):,} rows")
print(f"  - teams_df: {len(teams_df):,} rows")
print(f"  - teams_post_df: {len(teams_post_df):,} rows")

## Baseline Model - Coach Change Prediction (Before Feature Engineering)

Before building advanced features, let's establish a baseline model using only basic team performance metrics from the preprocessed data. This will help us measure the impact of our feature engineering efforts later.

In [ ]:
# Create target variable: coach changed next year
coaches_by_team_year_baseline = (
    coaches_df
    .groupby(['tmID', 'year'])['coachID']
    .agg(lambda x: ','.join(sorted(x.unique())))
    .reset_index()
    .sort_values(['tmID', 'year'])
)

coaches_by_team_year_baseline['next_year_coach'] = coaches_by_team_year_baseline.groupby('tmID')['coachID'].shift(-1)
coaches_by_team_year_baseline['coach_changed'] = (
    coaches_by_team_year_baseline['coachID'] != coaches_by_team_year_baseline['next_year_coach']
).astype(int)
coaches_by_team_year_baseline = coaches_by_team_year_baseline[coaches_by_team_year_baseline['next_year_coach'].notna()].copy()

# Create basic features from teams_df (current year only - no advanced engineering)
baseline_features = teams_df.copy()
baseline_features['win_pct'] = baseline_features['won'] / (baseline_features['won'] + baseline_features['lost'])
baseline_features['made_playoffs'] = (baseline_features['playoff'] == 1).astype(int)

# Merge with target
baseline_data = baseline_features.merge(
    coaches_by_team_year_baseline[['year', 'tmID', 'coach_changed']],
    on=['year', 'tmID'],
    how='inner'
)

# Select only basic performance features (no lagged, no trends, no advanced features)
feature_cols = ['won', 'lost', 'win_pct', 'rank', 'made_playoffs', 'playoff', 
                'o_pts', 'd_pts', 'o_reb', 'd_reb', 'o_asts', 'd_asts']

X_baseline = baseline_data[feature_cols].fillna(baseline_data[feature_cols].median())
y_baseline = baseline_data['coach_changed']

print("=== BASELINE DATASET ===")
print(f"Total samples: {len(X_baseline)}")
print(f"Features: {len(feature_cols)}")
print(f"Coach changes: {y_baseline.sum()} ({y_baseline.mean():.1%})")
print(f"No changes: {(y_baseline == 0).sum()} ({(y_baseline == 0).mean():.1%})")
print(f"\nFeature list: {feature_cols}")

In [ ]:
# Temporal split: Use earlier years for training, recent years for testing
# This prevents data leakage and simulates real-world prediction scenario
split_year = baseline_data['year'].quantile(0.8)

train_mask = baseline_data['year'] < split_year
test_mask = baseline_data['year'] >= split_year

X_train_base = X_baseline[train_mask]
X_test_base = X_baseline[test_mask]
y_train_base = y_baseline[train_mask]
y_test_base = y_baseline[test_mask]

print(f"=== TEMPORAL SPLIT ===")
print(f"Split year: {split_year:.0f}")
print(f"Training set: {len(X_train_base)} samples ({train_mask.sum()}) - Years < {split_year:.0f}")
print(f"Test set: {len(X_test_base)} samples ({test_mask.sum()}) - Years >= {split_year:.0f}")
print(f"\nTraining class distribution:")
print(f"  Coach changes: {y_train_base.sum()} ({y_train_base.mean():.1%})")
print(f"  No changes: {(y_train_base == 0).sum()} ({(y_train_base == 0).mean():.1%})")
print(f"\nTest class distribution:")
print(f"  Coach changes: {y_test_base.sum()} ({y_test_base.mean():.1%})")
print(f"  No changes: {(y_test_base == 0).sum()} ({(y_test_base == 0).mean():.1%})")

In [ ]:
# Standardize features for Logistic Regression
scaler_baseline = StandardScaler()
X_train_scaled = scaler_baseline.fit_transform(X_train_base)
X_test_scaled = scaler_baseline.transform(X_test_base)

# Train Logistic Regression (baseline model)
print("=== TRAINING LOGISTIC REGRESSION (BASELINE) ===")
lr_baseline = LogisticRegression(
    class_weight='balanced',  # Handle class imbalance
    random_state=42,
    max_iter=1000
)
lr_baseline.fit(X_train_scaled, y_train_base)

# Predictions
y_pred_lr = lr_baseline.predict(X_test_scaled)
y_pred_proba_lr = lr_baseline.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("\n--- Classification Report ---")
print(classification_report(y_test_base, y_pred_lr, target_names=['No Change', 'Coach Changed']))

print("\n--- Confusion Matrix ---")
cm_lr = confusion_matrix(y_test_base, y_pred_lr)
print(cm_lr)
print(f"\nTrue Negatives: {cm_lr[0,0]}, False Positives: {cm_lr[0,1]}")
print(f"False Negatives: {cm_lr[1,0]}, True Positives: {cm_lr[1,1]}")

# ROC-AUC
roc_auc_lr = roc_auc_score(y_test_base, y_pred_proba_lr)
print(f"\n--- ROC-AUC Score: {roc_auc_lr:.4f} ---")

# Feature importance (coefficients)
feature_importance_lr = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_baseline.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("\n--- Top 10 Most Important Features (Logistic Regression) ---")
display(feature_importance_lr.head(10))

In [ ]:
# Train Random Forest (baseline model)
print("=== TRAINING RANDOM FOREST (BASELINE) ===")
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5
)
rf_baseline.fit(X_train_base, y_train_base)

# Predictions
y_pred_rf = rf_baseline.predict(X_test_base)
y_pred_proba_rf = rf_baseline.predict_proba(X_test_base)[:, 1]

# Evaluation
print("\n--- Classification Report ---")
print(classification_report(y_test_base, y_pred_rf, target_names=['No Change', 'Coach Changed']))

print("\n--- Confusion Matrix ---")
cm_rf = confusion_matrix(y_test_base, y_pred_rf)
print(cm_rf)
print(f"\nTrue Negatives: {cm_rf[0,0]}, False Positives: {cm_rf[0,1]}")
print(f"False Negatives: {cm_rf[1,0]}, True Positives: {cm_rf[1,1]}")

# ROC-AUC
roc_auc_rf = roc_auc_score(y_test_base, y_pred_proba_rf)
print(f"\n--- ROC-AUC Score: {roc_auc_rf:.4f} ---")

# Feature importance
feature_importance_rf = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_baseline.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 10 Most Important Features (Random Forest) ---")
display(feature_importance_rf.head(10))

In [ ]:
# Visualize baseline model performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test_base, y_pred_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test_base, y_pred_proba_rf)

axes[0, 0].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_lr:.3f})', linewidth=2)
axes[0, 0].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_rf:.3f})', linewidth=2)
axes[0, 0].plot([0, 1], [0, 1], 'k--', label='Random Guess', linewidth=1)
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].set_title('ROC Curves - Baseline Models')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Feature Importance Comparison
top_n = 10
top_features_rf = feature_importance_rf.head(top_n)
axes[0, 1].barh(range(top_n), top_features_rf['importance'].values, color='steelblue', alpha=0.7)
axes[0, 1].set_yticks(range(top_n))
axes[0, 1].set_yticklabels(top_features_rf['feature'].values)
axes[0, 1].set_xlabel('Importance')
axes[0, 1].set_title('Top 10 Features - Random Forest Baseline')
axes[0, 1].invert_yaxis()
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Confusion Matrix - Logistic Regression
im1 = axes[1, 0].imshow(cm_lr, cmap='Blues', aspect='auto')
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_yticks([0, 1])
axes[1, 0].set_xticklabels(['No Change', 'Changed'])
axes[1, 0].set_yticklabels(['No Change', 'Changed'])
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_title('Confusion Matrix - Logistic Regression')
for i in range(2):
    for j in range(2):
        axes[1, 0].text(j, i, str(cm_lr[i, j]), ha='center', va='center', fontsize=14, fontweight='bold')

# 4. Confusion Matrix - Random Forest
im2 = axes[1, 1].imshow(cm_rf, cmap='Greens', aspect='auto')
axes[1, 1].set_xticks([0, 1])
axes[1, 1].set_yticks([0, 1])
axes[1, 1].set_xticklabels(['No Change', 'Changed'])
axes[1, 1].set_yticklabels(['No Change', 'Changed'])
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title('Confusion Matrix - Random Forest')
for i in range(2):
    for j in range(2):
        axes[1, 1].text(j, i, str(cm_rf[i, j]), ha='center', va='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("BASELINE MODEL SUMMARY")
print("="*60)
print(f"Logistic Regression ROC-AUC: {roc_auc_lr:.4f}")
print(f"Random Forest ROC-AUC:       {roc_auc_rf:.4f}")
print("\nThese scores will be our benchmark to beat with feature engineering!")
print("="*60)

In [ ]:
# Save baseline metrics for later comparison
baseline_metrics = {
    'logistic_regression': {
        'roc_auc': roc_auc_lr,
        'confusion_matrix': cm_lr.tolist(),
        'test_samples': len(y_test_base),
        'model': 'Logistic Regression (Baseline - No Feature Engineering)'
    },
    'random_forest': {
        'roc_auc': roc_auc_rf,
        'confusion_matrix': cm_rf.tolist(),
        'test_samples': len(y_test_base),
        'model': 'Random Forest (Baseline - No Feature Engineering)'
    }
}

print("✓ Baseline models trained and evaluated")
print("✓ Metrics saved for comparison after feature engineering")

## Coach Change Prediction - Feature Engineering

Based on EDA insights, we'll create features that predict coaching changes. Key findings:
- Poor team performance (low win%, missing playoffs, high losses) strongly correlates with coach changes
- Multi-year trends matter more than single-season statistics
- Team results are more predictive than offensive/defensive stats

### Step 1: Create Target Variable (Coach Changed Next Year)

In [ ]:
# Identify coach changes by comparing coaches year-to-year
coaches_by_team_year = (
    coaches_df
    .groupby(['tmID', 'year'])['coachID']
    .agg(lambda x: ','.join(sorted(x.unique())))  # Handle multiple coaches in same year
    .reset_index()
    .sort_values(['tmID', 'year'])
)

# Create next year's coach column
coaches_by_team_year['next_year_coach'] = coaches_by_team_year.groupby('tmID')['coachID'].shift(-1)

# Target: 1 if coach changed, 0 if same coach next year
coaches_by_team_year['coach_changed_next_year'] = (
    coaches_by_team_year['coachID'] != coaches_by_team_year['next_year_coach']
).astype(int)

# Remove rows where we don't have next year data (last year in dataset)
coaches_by_team_year = coaches_by_team_year[coaches_by_team_year['next_year_coach'].notna()].copy()

print(f"Total team-seasons: {len(coaches_by_team_year)}")
print(f"Coach changes: {coaches_by_team_year['coach_changed_next_year'].sum()}")
print(f"No change: {(coaches_by_team_year['coach_changed_next_year'] == 0).sum()}")
print(f"Change rate: {coaches_by_team_year['coach_changed_next_year'].mean():.1%}")

display(coaches_by_team_year.head(10))

### Step 2: Create Team Performance Features (Current Year)

In [ ]:
# Start with teams_df and create current season features
coach_pred_features = teams_df.copy()

# Basic performance metrics
coach_pred_features['win_pct'] = coach_pred_features['won'] / (coach_pred_features['won'] + coach_pred_features['lost'])
coach_pred_features['made_playoffs'] = (coach_pred_features['playoff'] == 1).astype(int)

# Home/Away performance
coach_pred_features['home_win_pct'] = coach_pred_features['homeW'] / (coach_pred_features['homeW'] + coach_pred_features['homeL'])
coach_pred_features['away_win_pct'] = coach_pred_features['awayW'] / (coach_pred_features['awayW'] + coach_pred_features['awayL'])
coach_pred_features['home_away_diff'] = coach_pred_features['home_win_pct'] - coach_pred_features['away_win_pct']

# Offensive and defensive efficiency (per EDA)
coach_pred_features['off_efficiency'] = coach_pred_features['o_pts'] / coach_pred_features['o_fga']
coach_pred_features['def_efficiency'] = coach_pred_features['d_pts'] / coach_pred_features['d_fga']
coach_pred_features['net_efficiency'] = coach_pred_features['off_efficiency'] - coach_pred_features['def_efficiency']

# Rebounding differential
coach_pred_features['reb_diff'] = coach_pred_features['o_reb'] - coach_pred_features['d_reb']
coach_pred_features['reb_diff_per_game'] = coach_pred_features['reb_diff'] / coach_pred_features['GP']

# Three-point shooting
coach_pred_features['three_pct'] = coach_pred_features['o_3pm'] / coach_pred_features['o_3pa']
coach_pred_features['opp_three_pct'] = coach_pred_features['d_3pm'] / coach_pred_features['d_3pa']

print("Current season features created")
print(f"Shape: {coach_pred_features.shape}")
display(coach_pred_features[['year', 'tmID', 'name', 'win_pct', 'made_playoffs', 'rank']].head())

### Step 3: Create Lagged Features (Previous Years Performance)

In [ ]:
# Sort by team and year to ensure proper lagging
coach_pred_features = coach_pred_features.sort_values(['tmID', 'year']).reset_index(drop=True)

# Previous year's performance (lag 1)
lag_features = ['win_pct', 'made_playoffs', 'rank', 'won', 'lost']

for feature in lag_features:
    coach_pred_features[f'{feature}_prev_year'] = coach_pred_features.groupby('tmID')[feature].shift(1)

# Two years ago performance (lag 2)
for feature in ['win_pct', 'made_playoffs', 'rank']:
    coach_pred_features[f'{feature}_2yr_ago'] = coach_pred_features.groupby('tmID')[feature].shift(2)

# Performance trends
coach_pred_features['win_pct_change'] = coach_pred_features['win_pct'] - coach_pred_features['win_pct_prev_year']
coach_pred_features['rank_change'] = coach_pred_features['rank'] - coach_pred_features['rank_prev_year']  # Negative = improvement
coach_pred_features['wins_change'] = coach_pred_features['won'] - coach_pred_features['won_prev_year']

# Multi-year trend (3-year moving average win%)
coach_pred_features['win_pct_3yr_avg'] = coach_pred_features.groupby('tmID')['win_pct'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Variance in performance (instability indicator)
coach_pred_features['win_pct_3yr_std'] = coach_pred_features.groupby('tmID')['win_pct'].transform(
    lambda x: x.rolling(window=3, min_periods=2).std()
)

print("Lagged features created")
print(f"Features with previous year data: {coach_pred_features['win_pct_prev_year'].notna().sum()}")
display(coach_pred_features[['year', 'tmID', 'win_pct', 'win_pct_prev_year', 'win_pct_change', 'rank', 'rank_change']].head(10))

### Step 3.5: Handle Missing Lagged Features (First/Second Year Teams)

In [ ]:
# Check missing values before imputation
print("=== Missing Values in Lagged Features (Before Imputation) ===")
lag_cols_to_check = [col for col in coach_pred_features.columns if 'prev_year' in col or '2yr_ago' in col or 'change' in col]
missing_before = coach_pred_features[lag_cols_to_check].isnull().sum()
print(missing_before[missing_before > 0])

# IMPROVED STRATEGY: Use year-specific league averages + indicator features
# This accounts for era differences and lets models learn expansion team patterns

# Step 1: Calculate league-wide statistics for each year
league_stats = coach_pred_features.groupby('year').agg({
    'win_pct': 'mean',
    'rank': 'median',
    'won': 'median',
    'lost': 'median',
    'made_playoffs': 'mean'
}).reset_index()
league_stats.columns = ['year', 'league_win_pct', 'league_rank', 'league_won', 'league_lost', 'league_playoff_rate']

# Merge league stats into main dataframe
coach_pred_features = coach_pred_features.merge(league_stats, on='year', how='left')

print(f"\n=== League Statistics by Year (Sample) ===")
display(league_stats.head(10))

# Step 2: Create indicator features BEFORE imputation (so models know which data is imputed)
coach_pred_features['is_first_year_team'] = (coach_pred_features.groupby('tmID').cumcount() == 0).astype(int)
coach_pred_features['is_second_year_team'] = (coach_pred_features.groupby('tmID').cumcount() == 1).astype(int)
coach_pred_features['team_years_in_league'] = coach_pred_features.groupby('tmID').cumcount() + 1

# Create "has prior data" indicators for key features
coach_pred_features['has_prev_year_data'] = coach_pred_features['win_pct_prev_year'].notna().astype(int)
coach_pred_features['has_2yr_ago_data'] = coach_pred_features['win_pct_2yr_ago'].notna().astype(int)

# Step 3: Impute missing values using year-specific league averages
# For win_pct lagged features: use year-specific league average
coach_pred_features['win_pct_prev_year'] = coach_pred_features['win_pct_prev_year'].fillna(
    coach_pred_features['league_win_pct']
)
coach_pred_features['win_pct_2yr_ago'] = coach_pred_features['win_pct_2yr_ago'].fillna(
    coach_pred_features['league_win_pct']
)

# For made_playoffs lagged features: use year-specific league playoff rate
coach_pred_features['made_playoffs_prev_year'] = coach_pred_features['made_playoffs_prev_year'].fillna(
    coach_pred_features['league_playoff_rate']
)
coach_pred_features['made_playoffs_2yr_ago'] = coach_pred_features['made_playoffs_2yr_ago'].fillna(
    coach_pred_features['league_playoff_rate']
)

# For rank lagged features: use year-specific league median
coach_pred_features['rank_prev_year'] = coach_pred_features['rank_prev_year'].fillna(
    coach_pred_features['league_rank']
)
coach_pred_features['rank_2yr_ago'] = coach_pred_features['rank_2yr_ago'].fillna(
    coach_pred_features['league_rank']
)

# For won/lost lagged features: use year-specific league medians
coach_pred_features['won_prev_year'] = coach_pred_features['won_prev_year'].fillna(
    coach_pred_features['league_won']
)
coach_pred_features['lost_prev_year'] = coach_pred_features['lost_prev_year'].fillna(
    coach_pred_features['league_lost']
)

# Step 4: Recalculate change features after imputation
coach_pred_features['win_pct_change'] = coach_pred_features['win_pct'] - coach_pred_features['win_pct_prev_year']
coach_pred_features['rank_change'] = coach_pred_features['rank'] - coach_pred_features['rank_prev_year']
coach_pred_features['wins_change'] = coach_pred_features['won'] - coach_pred_features['won_prev_year']

# Step 5: Fill 3-year std with 0 (no variance for new teams)
coach_pred_features['win_pct_3yr_std'] = coach_pred_features['win_pct_3yr_std'].fillna(0)

# Check missing values after imputation
print("\n=== Missing Values in Lagged Features (After Imputation) ===")
missing_after = coach_pred_features[lag_cols_to_check].isnull().sum()
if missing_after.sum() > 0:
    print(missing_after[missing_after > 0])
else:
    print("✓ All missing values handled!")

print(f"\n✓ Advanced Imputation Complete")
print(f"  - First-year teams: {coach_pred_features['is_first_year_team'].sum()}")
print(f"  - Second-year teams: {coach_pred_features['is_second_year_team'].sum()}")
print(f"  - Used year-specific league averages for era-adjusted imputation")
print(f"  - Created indicator features so models can learn expansion team patterns")

# Display sample of first-year teams to verify imputation
print("\n=== Sample: First-Year Teams (After Advanced Imputation) ===")
display(coach_pred_features[coach_pred_features['is_first_year_team'] == 1][
    ['year', 'tmID', 'win_pct', 'win_pct_prev_year', 'win_pct_change', 
     'rank', 'rank_prev_year', 'rank_change', 'is_first_year_team', 'has_prev_year_data']
].head(10))

### Step 4: Create Streak Features (Consecutive Performance Patterns)

In [ ]:
# Function to calculate consecutive streaks
def calculate_consecutive_streak(series, condition_func):
    """Calculate current consecutive streak where condition is True"""
    streaks = []
    for team in series.index.get_level_values(0).unique():
        team_data = series.loc[team]
        current_streak = 0
        team_streaks = []
        
        for val in team_data:
            if pd.isna(val):
                team_streaks.append(np.nan)
            elif condition_func(val):
                current_streak += 1
                team_streaks.append(current_streak)
            else:
                current_streak = 0
                team_streaks.append(0)
        
        streaks.extend(team_streaks)
    
    return streaks

# Set team as index temporarily for streak calculation
coach_pred_features_indexed = coach_pred_features.set_index(['tmID', 'year'])

# Consecutive losing seasons (win_pct < 0.5)
coach_pred_features['consecutive_losing_seasons'] = calculate_consecutive_streak(
    coach_pred_features_indexed['win_pct'],
    lambda x: x < 0.5
)

# Consecutive non-playoff seasons
coach_pred_features['consecutive_non_playoff_seasons'] = calculate_consecutive_streak(
    coach_pred_features_indexed['made_playoffs'],
    lambda x: x == 0
)

# Consecutive playoff appearances
coach_pred_features['consecutive_playoff_seasons'] = calculate_consecutive_streak(
    coach_pred_features_indexed['made_playoffs'],
    lambda x: x == 1
)

# Years since last playoff (0 if in playoffs this year)
coach_pred_features['years_since_playoff'] = coach_pred_features.groupby('tmID')['made_playoffs'].transform(
    lambda x: (x == 0).cumsum() - (x == 0).cumsum().where(x == 1).ffill().fillna(0)
).astype(int)

print("Streak features created")
display(coach_pred_features[['year', 'tmID', 'made_playoffs', 'consecutive_losing_seasons', 
                             'consecutive_non_playoff_seasons', 'years_since_playoff']].head(15))

### Step 5: Add Coach-Specific Features

In [ ]:
# Calculate coach tenure and experience
coaches_sorted = coaches_df.sort_values(['coachID', 'year'])

# Total years coaching (career experience)
coach_career_years = coaches_sorted.groupby('coachID')['year'].nunique().reset_index()
coach_career_years.columns = ['coachID', 'coach_career_years']

# Tenure with current team (consecutive years)
coaches_sorted['coach_tenure'] = coaches_sorted.groupby(['coachID', 'tmID']).cumcount() + 1

# Coach win percentage (career)
coach_stats = coaches_df.groupby('coachID').agg({
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()

coach_stats['coach_career_win_pct'] = coach_stats['won'] / (coach_stats['won'] + coach_stats['lost'])
coach_stats['coach_total_wins'] = coach_stats['won'] + coach_stats['post_wins']
coach_stats['coach_total_losses'] = coach_stats['lost'] + coach_stats['post_losses']

# Merge coach info with team data
# First, get primary coach per team-year (most wins)
primary_coaches = coaches_df.sort_values('won', ascending=False).groupby(['year', 'tmID']).first().reset_index()
primary_coaches = primary_coaches[['year', 'tmID', 'coachID', 'won', 'lost']]

# Add coach features to main dataset
coach_pred_features = coach_pred_features.merge(
    primary_coaches[['year', 'tmID', 'coachID']],
    on=['year', 'tmID'],
    how='left'
)

coach_pred_features = coach_pred_features.merge(
    coaches_sorted[['year', 'tmID', 'coachID', 'coach_tenure']],
    on=['year', 'tmID', 'coachID'],
    how='left'
)

coach_pred_features = coach_pred_features.merge(
    coach_career_years,
    on='coachID',
    how='left'
)

coach_pred_features = coach_pred_features.merge(
    coach_stats[['coachID', 'coach_career_win_pct', 'coach_total_wins']],
    on='coachID',
    how='left'
)

# Is this coach's first year with the team?
coach_pred_features['coach_is_new'] = (coach_pred_features['coach_tenure'] == 1).astype(int)

print("Coach features added")
print(f"Teams with coach data: {coach_pred_features['coachID'].notna().sum()}")
display(coach_pred_features[['year', 'tmID', 'coachID', 'coach_tenure', 'coach_career_win_pct', 'coach_is_new']].head(10))

### Step 6: Add Playoff Performance Features

In [ ]:
# Add playoff wins/losses from teams_post_df
playoff_perf = teams_post_df[['year', 'tmID', 'W', 'L']].copy()
playoff_perf.columns = ['year', 'tmID', 'playoff_wins', 'playoff_losses']

coach_pred_features = coach_pred_features.merge(
    playoff_perf,
    on=['year', 'tmID'],
    how='left'
)

# Fill NaN playoff stats with 0 (didn't make playoffs)
coach_pred_features['playoff_wins'] = coach_pred_features['playoff_wins'].fillna(0).astype(int)
coach_pred_features['playoff_losses'] = coach_pred_features['playoff_losses'].fillna(0).astype(int)

# Playoff round reached (encoded from firstRound, semis, finals columns)
coach_pred_features['reached_finals'] = coach_pred_features['finals'].notna().astype(int)
coach_pred_features['won_championship'] = (coach_pred_features['finals'] == 'W').astype(int)
coach_pred_features['reached_semis'] = coach_pred_features['semis'].notna().astype(int)
coach_pred_features['reached_first_round'] = coach_pred_features['firstRound'].notna().astype(int)

# Early playoff exit indicator (made playoffs but lost in first round)
coach_pred_features['early_playoff_exit'] = (
    (coach_pred_features['made_playoffs'] == 1) & 
    (coach_pred_features['firstRound'] == 'L')
).astype(int)

print("Playoff features added")
display(coach_pred_features[['year', 'tmID', 'made_playoffs', 'playoff_wins', 'reached_finals', 
                             'won_championship', 'early_playoff_exit']].head(10))

### Step 7: Merge with Target Variable and Create Final Dataset

In [ ]:
# Merge features with target variable
coach_change_dataset = coach_pred_features.merge(
    coaches_by_team_year[['year', 'tmID', 'coach_changed_next_year']],
    on=['year', 'tmID'],
    how='inner'  # Only keep rows where we have the target
)

print(f"Final dataset shape: {coach_change_dataset.shape}")
print(f"Total samples: {len(coach_change_dataset)}")
print(f"Coach changes: {coach_change_dataset['coach_changed_next_year'].sum()}")
print(f"No changes: {(coach_change_dataset['coach_changed_next_year'] == 0).sum()}")
print(f"Class balance: {coach_change_dataset['coach_changed_next_year'].mean():.1%} changed")

# Check for missing values in key features
print("\n=== Missing Values in Key Features ===")
key_features = [
    'win_pct', 'rank', 'made_playoffs', 'won', 'lost',
    'win_pct_prev_year', 'rank_prev_year', 'made_playoffs_prev_year',
    'win_pct_change', 'rank_change', 
    'consecutive_losing_seasons', 'consecutive_non_playoff_seasons',
    'coach_tenure', 'coach_career_win_pct'
]
missing_summary = coach_change_dataset[key_features].isnull().sum()
display(missing_summary[missing_summary > 0])

print(f"\n=== Sample of Final Dataset ===")
display(coach_change_dataset[['year', 'tmID', 'name', 'win_pct', 'rank', 'made_playoffs', 
                               'win_pct_prev_year', 'rank_change', 'consecutive_losing_seasons',
                               'coach_tenure', 'coach_changed_next_year']].head(15))

### Step 8: Feature Correlation Analysis with Target

In [ ]:
# Identify all numeric features for correlation analysis
numeric_cols = coach_change_dataset.select_dtypes(include=[np.number]).columns.tolist()

# Exclude identifiers and the target itself
exclude_cols = ['year', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW', 'confL', 
                'coach_changed_next_year', 'min', 'attend']
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

# Calculate correlations with target
correlations = coach_change_dataset[feature_cols + ['coach_changed_next_year']].corr()['coach_changed_next_year'].drop('coach_changed_next_year')

# Sort by absolute correlation
correlations_sorted = correlations.reindex(correlations.abs().sort_values(ascending=False).index)

# Display top correlations
print("=== TOP 20 FEATURES BY CORRELATION WITH COACH CHANGE ===\n")
display(correlations_sorted.head(20).to_frame('Correlation'))

# Visualize top features
plt.figure(figsize=(12, 10))
top_n = 25
colors = ['red' if x > 0 else 'green' for x in correlations_sorted.head(top_n).values]
plt.barh(range(top_n), correlations_sorted.head(top_n).values, color=colors, alpha=0.7)
plt.yticks(range(top_n), correlations_sorted.head(top_n).index, fontsize=9)
plt.xlabel('Correlation with Coach Change (Next Year)', fontsize=11)
plt.title('Top 25 Features Correlated with Coaching Changes\n(Red = Positive, Green = Negative)', fontsize=12)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)

# Add correlation values
for i, v in enumerate(correlations_sorted.head(top_n).values):
    plt.text(v + (0.005 if v > 0 else -0.005), i, f'{v:.3f}',
             va='center', ha='left' if v > 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\n=== INTERPRETATION ===")
print("Positive correlation: Higher values → More likely to change coach")
print("Negative correlation: Higher values → Less likely to change coach")

### Step 9: Export Final Coach Change Dataset

In [ ]:
output_path = '../merged_datasets/coach_change_dataset.csv'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

coach_change_dataset.to_csv(output_path, index=False)

print(f"✓ Dataset saved to: {output_path}")
print(f"  Rows: {len(coach_change_dataset)}")
print(f"  Columns: {coach_change_dataset.shape[1]}")
print(f"  Target variable: coach_changed_next_year")
print(f"  Positive class: {coach_change_dataset['coach_changed_next_year'].sum()} ({coach_change_dataset['coach_changed_next_year'].mean():.1%})")

# Summary of feature categories
print("\n=== FEATURE CATEGORIES ===")
print(f"Current season performance: win_pct, rank, made_playoffs, won, lost, etc.")
print(f"Historical performance: *_prev_year, *_2yr_ago features")
print(f"Performance trends: win_pct_change, rank_change, wins_change")
print(f"Streak features: consecutive_losing_seasons, consecutive_non_playoff_seasons, years_since_playoff")
print(f"Coach features: coach_tenure, coach_career_win_pct, coach_is_new")
print(f"Playoff features: playoff_wins, reached_finals, won_championship, early_playoff_exit")
print(f"Efficiency metrics: off_efficiency, def_efficiency, net_efficiency, reb_diff")

print("\n=== READY FOR MODELING ===")
print("Next steps:")
print("1. Handle remaining missing values (first-year teams without lagged features)")
print("2. Split into train/validation/test sets (temporal split recommended)")
print("3. Address class imbalance if needed (SMOTE, class weights, etc.)")
print("4. Feature selection/importance analysis")
print("5. Train classification models (Logistic Regression, Random Forest, XGBoost)")